In [1]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader

# Detectar se está no Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Executando no Google Colab")
except:
    IN_COLAB = False
    print("✓ Executando localmente")

# Configurar caminho do projeto
if IN_COLAB:
    # Clonar repositório do GitHub
    if not os.path.exists('/content/ufc-easytpp'):
        print("Clonando repositório...")
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    else:
        print("✓ Repositório já clonado")

    project_root = '/content/ufc-easytpp'
else:
    # Caminho local (Mac)
    project_root = '/Users/hugoramossoares/Sites/EasyTemporalPointProcess'

# Adicionar ao path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✓ Project root: {project_root}")

# Importar modelos
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_rothp_hybrid import RoTHPHybrid
from easy_tpp.model.torch_model.torch_thp import THP
from easy_tpp.model.torch_model.torch_nhp import NHP
from easy_tpp.model.torch_model.torch_thp_expdecay import THPExpDecay
from easy_tpp.model.torch_model.torch_attnhp import AttNHP

print("✓ Bibliotecas carregadas.")

# Verificar GPU
print(f"✓ GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ Rodando em CPU - Vá em Runtime → Change runtime type → GPU")

✓ Executando localmente
✓ Project root: /Users/hugoramossoares/Sites/EasyTemporalPointProcess
✓ Bibliotecas carregadas.
✓ GPU disponível: False
⚠ Rodando em CPU - Vá em Runtime → Change runtime type → GPU


In [2]:

# =============================================================================
# CARREGAR DATASET (Retweet via HuggingFace)
# =============================================================================
print("Carregando dataset 'easytpp/retweet'...")
dataset = load_dataset("easytpp/retweet")

train_data = dataset['train']
dev_data = dataset['validation']
test_data = dataset['test']

print(f"\nDataset Retweet carregado!")
print(f"Treino: {len(train_data)} sequências")
print(f"Dev:    {len(dev_data)} sequências")
print(f"Teste:  {len(test_data)} sequências")

# Verificar metadados do primeiro exemplo
sample = train_data[0]
print("\nExemplo de sequência:")
print(f"Event Types: {sample['type_event'][:10]}...")
print(f"Time deltas: {sample['time_since_last_event'][:10]}...")
print(f"Dim Process (Num Event Types): {sample['dim_process']}")


Carregando dataset 'easytpp/retweet'...

Dataset Retweet carregado!
Treino: 20000 sequências
Dev:    2000 sequências
Teste:  2000 sequências

Exemplo de sequência:
Event Types: [1, 1, 1, 1, 0, 1, 0, 0, 0, 0]...
Time deltas: [0.0, 1.0, 3.0, 4.0, 0.0, 2.0, 3.0, 2.0, 2.0, 1.0]...
Dim Process (Num Event Types): 3


In [3]:

# =============================================================================
# PREPARAÇÃO DE BATCH (Adaptado para formato HuggingFace)
# =============================================================================
def collate_fn(batch_list):
    """Converte lista de dicts (HF) para tensores de batch do EasyTPP."""
    time_seqs = []
    time_delta_seqs = []
    type_seqs = []

    max_len = 0
    for item in batch_list:
        # Converter listas python para tensores, se necessário
        ts = item['time_since_start']
        td = item['time_since_last_event']
        ev = item['type_event']

        if len(ts) > max_len:
            max_len = len(ts)

        time_seqs.append(torch.tensor(ts, dtype=torch.float32))
        time_delta_seqs.append(torch.tensor(td, dtype=torch.float32))
        type_seqs.append(torch.tensor(ev, dtype=torch.long))

    batch_size = len(batch_list)

    # Padding
    pad_time = torch.zeros(batch_size, max_len)
    pad_delta = torch.zeros(batch_size, max_len)
    pad_type = torch.zeros(batch_size, max_len, dtype=torch.long)
    attention_mask = torch.zeros(batch_size, max_len, max_len) # mask de atenção pad
    batch_non_pad_mask = torch.zeros(batch_size, max_len)

    for i in range(batch_size):
        l = len(time_seqs[i])
        pad_time[i, :l] = time_seqs[i]
        pad_delta[i, :l] = time_delta_seqs[i]
        pad_type[i, :l] = type_seqs[i]
        batch_non_pad_mask[i, :l] = 1

        # Causal Mask (Triangular Superior)
        causal_mask = torch.tril(torch.ones(l, l))
        attention_mask[i, :l, :l] = causal_mask

    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

# Teste rápido do Collate
sample_batch = [train_data[0], train_data[1]]
batch_tensors = collate_fn(sample_batch)
print("Batch Tensors Shapes:", [t.shape for t in batch_tensors])


Batch Tensors Shapes: [torch.Size([2, 88]), torch.Size([2, 88]), torch.Size([2, 88]), torch.Size([2, 88]), torch.Size([2, 88, 88])]


In [4]:
# =============================================================================
# EXTRAÇÃO AUTOMÁTICA DE METADADOS
# =============================================================================
def get_dataset_specs(data):
    max_type = 0
    for item in data:
        types = item['type_event']
        if len(types) > 0:
            current_max = max(types)
            if current_max > max_type:
                max_type = current_max

    # Assumindo IDs sequenciais 0, 1, ..., max_type
    num_types = max_type + 1
    return num_types

# Calcular specs baseados no treino
NUM_EVENT_TYPES = get_dataset_specs(train_data)
PAD_TOKEN_ID = NUM_EVENT_TYPES
NUM_EVENT_TYPES_PAD = NUM_EVENT_TYPES + 1

print(f"\nMetadados detectados automaticamente:")
print(f"Num Event Types: {NUM_EVENT_TYPES}")
print(f"Pad Token ID:    {PAD_TOKEN_ID}")
print(f"Total Vocab:     {NUM_EVENT_TYPES_PAD}")

# Estatísticas de comprimento de sequência
seq_lengths = [len(item['time_since_start']) for item in train_data]
print(f"\nEstatísticas de Comprimento de Sequência (Treino):")
print(f"  Média:   {np.mean(seq_lengths):.1f}")
print(f"  Mediana: {np.median(seq_lengths):.1f}")
print(f"  Min:     {np.min(seq_lengths)}")
print(f"  Max:     {np.max(seq_lengths)}")
print(f"  Std:     {np.std(seq_lengths):.1f}")

# Calcular escala temporal para normalização
all_deltas = []
for item in train_data:
    td = item['time_since_last_event']
    all_deltas.extend([d for d in td if d > 0])

TIME_SCALE = np.mean(all_deltas)
print(f"\nEscala Temporal (média dos deltas no treino):")
print(f"  Mean delta:   {TIME_SCALE:.4f}")
print(f"  Median delta: {np.median(all_deltas):.4f}")
print(f"  Max delta:    {np.max(all_deltas):.4f}")
print(f"  Min delta:    {np.min(all_deltas):.4f}")
print(f"  >> Todos os tempos serão divididos por {TIME_SCALE:.4f} para normalização")



Metadados detectados automaticamente:
Num Event Types: 3
Pad Token ID:    3
Total Vocab:     4

Estatísticas de Comprimento de Sequência (Treino):
  Média:   108.8
  Mediana: 90.0
  Min:     50
  Max:     264
  Std:     54.5

Escala Temporal (média dos deltas no treino):
  Mean delta:   2704.2936
  Median delta: 40.0000
  Max delta:    582928.0000
  Min delta:    1.0000
  >> Todos os tempos serão divididos por 2704.2936 para normalização


In [6]:

# =============================================================================
# CONFIGURAÇÃO DO MODELO
# =============================================================================


class ThinningConfig:
    def __init__(self, dtime_max=5.0, num_sample=200, num_exp=500):
        self.num_sample = num_sample     # Amostras para calcular a integral/predição
        self.num_exp = num_exp           # Amostras para o algoritmo de thinning
        self.over_sample_rate = 10.0
        self.patience_counter = 5
        self.num_samples_boundary = 20
        self.dtime_max = dtime_max       # Horizonte máximo de tempo para predição (em unidades normalizadas)


class ModelConfig:
    def __init__(self, num_types, pad_id, num_types_pad, hidden_size=64, num_heads=4, num_layers=2):
        self.hidden_size = hidden_size
        self.time_emb_size = hidden_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout_rate = 0.1
        self.use_ln = True

        # Specs Automáticos
        self.num_event_types = num_types
        self.num_event_types_pad = num_types_pad
        self.pad_token_id = pad_id

        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = -1

        # IMPORTANTE: Configuração de Thinning para habilitar .predict()
        self.thinning = ThinningConfig()

        # NHP Specifics
        self.model_specs = {'beta': 1.0, 'bias': True}

    def __str__(self):
        return str(self.__dict__)


def compute_metrics(model, test_ds):
    model.eval()

    total_acc = 0
    total_rmse = 0
    total_events = 0

    # Avaliar em subset do teste para ser rápido (ex: 100 sequências)
    subset_size = min(100, len(test_ds))
    subset = [test_ds[i] for i in range(subset_size)]

    # Processar um por um ou em batches pequenos para evitar estourar memória na amostragem
    batch_size_eval = 10

    with torch.no_grad():
        for i in range(0, subset_size, batch_size_eval):
            batch_list = subset[i:i+batch_size_eval]
            batch = collate_fn_shifted(batch_list)

            # Unpack batch para pegar targets
            # batch = (time_seqs, time_delta_seqs, type_seqs, batch_non_pad_mask, attention_mask)
            _, time_delta_target, type_target, mask_target, _ = batch

            # Predict
            # dtimes_pred: [B, L-1] (deltas previstos)
            # types_pred:  [B, L-1] (tipos previstos)
            dtimes_pred, types_pred = model.predict_one_step_at_every_event(batch)

            # Ajustar targets (removemos o primeiro evento pois não prevemos o passado,
            # e o predict já remove o último evento da entrada para prever o próximo)
            # O output do predict alinha com o target do índice 1 ao fim.

            target_types = type_target[:, 1:]
            target_deltas = time_delta_target[:, 1:]
            target_mask = mask_target[:, 1:]

            # Calcular Acurácia (Type)
            correct = (types_pred == target_types) * target_mask
            total_acc += correct.sum().item()

            # Calcular RMSE (Time)
            se = ((dtimes_pred - target_deltas) ** 2) * target_mask
            total_rmse += se.sum().item()

            total_events += target_mask.sum().item()

    avg_acc = total_acc / (total_events + 1e-9)
    avg_rmse = np.sqrt(total_rmse / (total_events + 1e-9))

    return avg_acc, avg_rmse

def train_eval_loop(model_class, name, config, train_ds, test_ds, epochs=50, batch_size=64, lr=5e-4, patience=15):
    import time as time_module
    print(f"\n>>> Treinando: {name}")
    torch.manual_seed(42)
    model = model_class(config)

    # Contar parâmetros
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parâmetros treináveis: {num_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    history_nll = []
    best_nll = float('inf')
    patience_counter = 0
    clip_val = 1.0

    t_train_start = time_module.time()

    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        total_events = 0

        indices = np.random.permutation(len(train_ds))
        max_batches = 100

        for i in range(0, len(indices), batch_size):
            if i // batch_size > max_batches: break
            batch_idx = indices[i:i+batch_size]
            batch_list = [train_ds[int(k)] for k in batch_idx]

            try:
                batch_data = collate_fn_shifted(batch_list)
                optimizer.zero_grad()
                loss, num_events = model.loglike_loss(batch_data)

                if torch.isnan(loss) or torch.isinf(loss):
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_val)
                optimizer.step()

                total_loss += loss.item()
                total_events += num_events
            except Exception as e:
                continue

        nll_train = total_loss / (total_events + 1e-9)

        # Validation NLL (batches menores para evitar padding excessivo)
        model.eval()
        val_loss_total = 0
        val_events_total = 0
        with torch.no_grad():
            test_subset = [test_ds[i] for i in range(min(200, len(test_ds)))]
            val_batch_size = 32
            for vi in range(0, len(test_subset), val_batch_size):
                val_batch = collate_fn_shifted(test_subset[vi:vi+val_batch_size])
                vl, vn = model.loglike_loss(val_batch)
                if not (torch.isnan(vl) or torch.isinf(vl)):
                    val_loss_total += vl.item()
                    val_events_total += vn
        nll_test = val_loss_total / (val_events_total + 1e-9)

        print(f"  Ep {epoch} | Train NLL: {nll_train:.4f} | Val NLL: {nll_test:.4f}")
        history_nll.append(nll_test)

        scheduler.step(nll_test)

        if nll_test < best_nll:
            best_nll = nll_test
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  >>> Early Stopping.")
                break

    t_train_end = time_module.time()
    train_time = t_train_end - t_train_start
    num_epochs_run = len(history_nll)

    # Final Metrics Calculation
    print(f"  Calculando métricas finais (Acc, RMSE)...")
    t_eval_start = time_module.time()
    final_acc, final_rmse = compute_metrics(model, test_ds)
    t_eval_end = time_module.time()
    eval_time = t_eval_end - t_eval_start

    total_time = train_time + eval_time
    print(f"  >>> Resultado Final {name}: Acc={final_acc:.4f}, RMSE={final_rmse:.4f}")
    print(f"  >>> Tempo: Treino={train_time:.1f}s ({num_epochs_run} épocas, {train_time/num_epochs_run:.1f}s/ep) | Eval={eval_time:.1f}s | Total={total_time:.1f}s")
    print(f"  >>> Parâmetros: {num_params:,}")

    return history_nll, final_acc, final_rmse, train_time, eval_time, num_params, model


In [7]:

# =============================================================================
# PREPARAÇÃO DE BATCH COM NORMALIZAÇÃO TEMPORAL
# =============================================================================

SHIFT_VAL = 0.0

def collate_fn_shifted(batch_list):
    time_seqs = []
    time_delta_seqs = []
    type_seqs = []

    max_len = 0
    for item in batch_list:
        ts = item['time_since_start']
        td = item['time_since_last_event']
        ev = item['type_event']

        if len(ts) > max_len:
            max_len = len(ts)

        # NORMALIZAÇÃO TEMPORAL
        # 1. Subtrair t[0] para começar em 0
        ts_raw = torch.tensor(ts, dtype=torch.float64) + SHIFT_VAL
        ts_normalized = ts_raw - ts_raw[0]

        # 2. Dividir pela escala temporal (mean delta) para trazer valores para ~O(1)
        #    Isso é CRUCIAL para o THP, que usa decay LINEAR: factor * delta_t
        #    Sem normalização, delta_t pode ser ~1000+, explodindo a intensidade
        ts_final = (ts_normalized / TIME_SCALE).to(torch.float32)
        td_final = (torch.tensor(td, dtype=torch.float64) / TIME_SCALE).to(torch.float32)

        time_seqs.append(ts_final)
        time_delta_seqs.append(td_final)
        type_seqs.append(torch.tensor(ev, dtype=torch.long))

    batch_size = len(batch_list)

    pad_time = torch.zeros(batch_size, max_len)
    pad_delta = torch.zeros(batch_size, max_len)
    pad_type = torch.zeros(batch_size, max_len, dtype=torch.long)
    attention_mask = torch.zeros(batch_size, max_len, max_len)
    batch_non_pad_mask = torch.zeros(batch_size, max_len)

    for i in range(batch_size):
        l = len(time_seqs[i])
        pad_time[i, :l] = time_seqs[i]
        pad_delta[i, :l] = time_delta_seqs[i]
        pad_type[i, :l] = type_seqs[i]
        batch_non_pad_mask[i, :l] = 1

        # MÁSCARA DE ATENÇÃO: Combina causal + padding (convenção EasyTPP: 1 = BLOQUEAR)
        # 1. Causal: triu(k=1) bloqueia posições futuras
        causal_mask = torch.triu(torch.ones(max_len, max_len), diagonal=1)
        # 2. Padding: bloqueia COLUNAS de padding (key positions que são padding)
        causal_mask[:, l:] = 1
        # 3. Padding rows: queries de padding bloqueiam tudo
        causal_mask[l:, :] = 1
        attention_mask[i] = causal_mask

    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

print(f"Collate function configurada com SHIFT = {SHIFT_VAL}")


Collate function configurada com SHIFT = 0.0


In [8]:

config = ModelConfig(
    num_types=NUM_EVENT_TYPES,
    pad_id=PAD_TOKEN_ID,
    num_types_pad=NUM_EVENT_TYPES_PAD,
    hidden_size=64,
    num_heads=2,
    num_layers=2
)

results = {}

# Executar cada modelo individualmente (comente/descomente conforme necessário)

# --- NHP ---
hist, acc, rmse, t_train, t_eval, n_params, trained_model = train_eval_loop(NHP, "NHP (RNN)", config, train_data, test_data)
results["NHP (RNN)"] = {'hist': hist, 'acc': acc, 'rmse': rmse, 'train_time': t_train, 'eval_time': t_eval, 'total_time': t_train + t_eval, 'num_params': n_params, 'model': trained_model}

# --- THP ---
# hist, acc, rmse, t_train, t_eval, n_params, trained_model = train_eval_loop(THP, "THP Tradicional", config, train_data, test_data)
# results["THP Tradicional"] = {'hist': hist, 'acc': acc, 'rmse': rmse, 'train_time': t_train, 'eval_time': t_eval, 'total_time': t_train + t_eval, 'num_params': n_params, 'model': trained_model}

# --- THP-ExpDecay (PROPOSTO: Transformer + Exponential Decay) ---
# hist, acc, rmse, t_train, t_eval, n_params, trained_model = train_eval_loop(THPExpDecay, "THP-ExpDecay (Proposto)", config, train_data, test_data)
# results["THP-ExpDecay (Proposto)"] = {'hist': hist, 'acc': acc, 'rmse': rmse, 'train_time': t_train, 'eval_time': t_eval, 'total_time': t_train + t_eval, 'num_params': n_params, 'model': trained_model}

# --- RoTHP ---
# hist, acc, rmse, t_train, t_eval, n_params, trained_model = train_eval_loop(RoTHP, "RoTHP (Transformer)", config, train_data, test_data)
# results["RoTHP (Transformer)"] = {'hist': hist, 'acc': acc, 'rmse': rmse, 'train_time': t_train, 'eval_time': t_eval, 'total_time': t_train + t_eval, 'num_params': n_params, 'model': trained_model}

# --- RoTHP Híbrido ---
#hist, acc, rmse, t_train, t_eval, n_params, trained_model = train_eval_loop(RoTHPHybrid, "RoTHP Híbrido", config, train_data, test_data)
#results["RoTHP Híbrido"] = {'hist': hist, 'acc': acc, 'rmse': rmse, 'train_time': t_train, 'eval_time': t_eval, 'total_time': t_train + t_eval, 'num_params': n_params, 'model': trained_model}



>>> Treinando: NHP (RNN)
  Parâmetros treináveis: 58,246
  Ep 1 | Train NLL: -0.1966 | Val NLL: -0.9651
  Ep 2 | Train NLL: -1.2585 | Val NLL: -1.4498
  Ep 3 | Train NLL: -1.5774 | Val NLL: -1.6446
  Ep 4 | Train NLL: -1.7069 | Val NLL: -1.7296
  Ep 5 | Train NLL: -1.7716 | Val NLL: -1.7884
  Ep 6 | Train NLL: -1.8230 | Val NLL: -1.8400
  Ep 7 | Train NLL: -1.8735 | Val NLL: -1.8881
  Ep 8 | Train NLL: -1.9237 | Val NLL: -1.9328
  Ep 9 | Train NLL: -1.9574 | Val NLL: -1.9755
  Ep 10 | Train NLL: -1.9939 | Val NLL: -2.0163
  Ep 11 | Train NLL: -2.0451 | Val NLL: -2.0555
  Ep 12 | Train NLL: -2.0859 | Val NLL: -2.0936
  Ep 13 | Train NLL: -2.1357 | Val NLL: -2.1306
  Ep 14 | Train NLL: -2.1654 | Val NLL: -2.1672
  Ep 15 | Train NLL: -2.1888 | Val NLL: -2.2028
  Ep 16 | Train NLL: -2.2146 | Val NLL: -2.2370
  Ep 17 | Train NLL: -2.2621 | Val NLL: -2.2716
  Ep 18 | Train NLL: -2.3054 | Val NLL: -2.3051
  Ep 19 | Train NLL: -2.3518 | Val NLL: -2.3381
  Ep 20 | Train NLL: -2.3573 | Val NLL: